# 01 · Exploración de datos

Comprobación de la disponibilidad, cobertura y calidad de las fuentes antes de cualquier estimación. Si faltan ficheros, ver `data/README.md`.

In [1]:
# Configuración: raíz del repositorio en el path
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

# Para comprobar el código sin datos reales: os.environ["INMIG_DEMO"] = "1"
from src import config
print("MODO DEMO (DATOS SINTÉTICOS)" if config.is_demo() else "Datos reales")

Datos reales


In [2]:
from src import data_collection as dcol
dcol.check_manual_files()

,fichero,presente,descripcion
0,poblacion_inmigrante.csv,True,OBLIGATORIO - INE Padrón: nacidos en el extran...
1,precio_vivienda.csv,True,OBLIGATORIO - MIVAU: valor tasado medio vivien...
2,extranjeros_nacionalidad.csv,True,OPCIONAL (necesario para IV shift-share) - INE...
3,viviendas_terminadas.csv,False,OPCIONAL - MIVAU: viviendas libres terminadas ...


## Construcción y validación del panel provincia × año

In [3]:
from src import data_cleaning as dc
panel = dc.load_panel()
print(panel.shape)
panel.head()

(1300, 31)


,provincia_cod,anio,precio_m2,precio_m2_t4,anio_padron,nacidos_extranjero,extranjeros,poblacion_total,poblacion,ln_precio,...,d_pct_extranjeros,flujo_extranjeros,flujo_nacidos_ext_l1,flujo_nacidos_ext_l2,flujo_extranjeros_l1,flujo_extranjeros_l2,pct_nacidos_ext_l1,pct_nacidos_ext_l2,ccaa,provincia
0,01,1997,1013.650,1039.5,1998,4603,2460,284595,284595,6.921313,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,País Vasco,Araba/Álava
1,01,1998,1151.875,1272.4,1999,4947,2801,285748,285748,7.049146,...,0.115848,0.119819,NaN,NaN,NaN,NaN,1.617386,NaN,País Vasco,Araba/Álava
2,01,1999,1392.325,1467.6,2000,6108,3818,286497,286497,7.238730,...,0.352415,0.355908,0.120874,NaN,0.119819,NaN,1.731246,1.617386,País Vasco,Araba/Álava
3,01,2000,1475.250,1554.6,2001,7850,5462,288793,288793,7.296583,...,0.558671,0.573828,0.406302,0.120874,0.355908,0.119819,2.131959,1.731246,País Vasco,Araba/Álava
4,01,2001,1653.375,1712.2,2002,10573,8031,291860,291860,7.410574,...,0.860342,0.889564,0.608034,0.406302,0.573828,0.355908,2.718210,2.131959,País Vasco,Araba/Álava


In [4]:
rep = dc.validate_panel(panel)
{k: v for k, v in rep.items() if k != 'missing_pct'}

{'provincias': 52, 'anios': (1997, 2021), 'observaciones': 1300}

## Datos ausentes por variable (%)

Las variables con muchos ausentes limitan el periodo común de estimación.

In [5]:
import pandas as pd
pd.Series(rep['missing_pct']).sort_values(ascending=False).head(25)

flujo_extranjeros_l2    12.0
flujo_nacidos_ext_l2    12.0
flujo_extranjeros_l1     8.0
flujo_nacidos_ext_l1     8.0
pct_nacidos_ext_l2       8.0
dln_precio_t4            6.5
dln_precio               6.5
g_nacidos_ext            4.0
flujo_nativos            4.0
flujo_nacidos_ext        4.0
d_pct_nacidos_ext        4.0
flujo_extranjeros        4.0
d_pct_extranjeros        4.0
pct_nacidos_ext_l1       4.0
g_poblacion              4.0
ln_precio_t4             1.8
precio_m2_t4             1.8
precio_m2                1.8
ln_precio                1.8
anio                     0.0
anio_padron              0.0
nacidos_extranjero       0.0
extranjeros              0.0
pct_nacidos_ext          0.0
poblacion_total          0.0
dtype: float64

## Cobertura temporal por variable

In [6]:
cov = panel.groupby('anio')[[c for c in ['precio_m2','nacidos_extranjero','extranjeros','pib_meur','tasa_paro_ccaa','viviendas_terminadas'] if c in panel]].count()
cov

,precio_m2,nacidos_extranjero,extranjeros
anio,,,
1997,50,52,52
1998,50,52,52
1999,50,52,52
2000,50,52,52
2001,50,52,52
2002,50,52,52
2003,50,52,52
2004,52,52,52
2005,52,52,52
